# ChakraTS on GIFT-Eval (hosted API)
The model is served at `https://api.yhatlabs.com/v1/forecast`. Set `YHAT_API_KEY` (evaluation keys are issued on request, see the model card). Each request carries a batch of series (history + horizon); the
response is 9 quantiles (0.1..0.9) per step. Evaluation below is the standard GIFT-Eval protocol (gluonts `evaluate_forecasts`, axis=None,
NaN labels masked). Multivariate datasets are evaluated per variate (`to_univariate=True`).

## Setup
1. Clone [gift-eval](https://github.com/SalesforceAIResearch/gift-eval), `pip install -e .`, download the benchmark data and set `GIFT_EVAL` as its README describes.
2. Copy this notebook and `gift_configs.csv` into gift-eval's `notebooks/` folder (it reads `dataset_properties.json` from there).
3. Set `YHAT_API_KEY`, then run all cells. The output is `results/<MODEL>/all_results.csv` in the leaderboard format.


In [ ]:
import json, os, csv, numpy as np, pandas as pd, requests
from gift_eval.data import Dataset
from gluonts.model.forecast import QuantileForecast
from gluonts.model import evaluate_forecasts
from gluonts.ev.metrics import MSE, MAE, MASE, MAPE, SMAPE, MSIS, RMSE, NRMSE, ND, MeanWeightedSumQuantileLoss
from gluonts.time_feature import get_seasonality
MODEL = "ChakraTS"; API = os.environ.get("YHAT_API_URL", "https://api.yhatlabs.com/v1/forecast"); KEY = os.environ["YHAT_API_KEY"]
LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
METRICS = [MSE(forecast_type="mean"), MSE(forecast_type=0.5), MAE(), MASE(), MAPE(), SMAPE(), MSIS(), RMSE(), NRMSE(), ND(), MeanWeightedSumQuantileLoss(quantile_levels=LEVELS)]
props = json.load(open("dataset_properties.json"))


In [ ]:
def forecast_batch(histories, horizon, freq):
    body = {"series": [{"id": str(i), "values": [None if not np.isfinite(v) else float(v) for v in h]} for i, h in enumerate(histories)], "horizon": horizon, "freq": freq}
    r = requests.post(API, json=body, headers={"Authorization": f"Bearer {KEY}"}, timeout=600); r.raise_for_status()
    return [np.asarray(f["quantiles"], dtype=np.float32) for f in r.json()["forecasts"]]  # each [horizon, 9]

def run(ds_name, term):
    ds = Dataset(name=ds_name, term=term, to_univariate=(props[ds_name.split("/")[0].lower()]["num_variates"] > 1))
    test = ds.test_data; season = get_seasonality(ds.freq); fc = []
    inputs = list(test.input); labels = list(test.label)
    for i in range(0, len(inputs), 64):
        chunk = inputs[i:i + 64]; qs = forecast_batch([np.asarray(e["target"], dtype=np.float32) for e in chunk], ds.prediction_length, ds.freq)
        for e, lab, q in zip(chunk, labels[i:i + 64], qs): fc.append(QuantileForecast(forecast_arrays=q.T, start_date=lab["start"], forecast_keys=[str(l) for l in LEVELS], item_id=e["item_id"]))
    res = evaluate_forecasts(fc, test_data=test, metrics=METRICS, axis=None, batch_size=1024, mask_invalid_label=True, allow_nan_forecast=False, seasonality=season)
    return res


In [ ]:
rows = []
for cfg in csv.DictReader(open("gift_configs.csv")):  # the 97 dataset/freq/term configs
    ds_name, term = cfg["dataset"].rsplit("/", 1); res = run(ds_name, term); key = ds_name.split("/")[0].lower()
    rows.append({"dataset": cfg["dataset"], "model": MODEL, **{f"eval_metrics/{c}": float(res[c].iloc[0]) for c in res.columns}, "domain": props[key]["domain"], "num_variates": props[key]["num_variates"]})
os.makedirs(f"results/{MODEL}", exist_ok=True); pd.DataFrame(rows).to_csv(f"results/{MODEL}/all_results.csv", index=False)
